In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction==2026.11.0 --extra-index-url https://download.easydiffraction.org/pypi/simple/


# Si on SEPD

&nbsp; &nbsp; &nbsp; &nbsp; &nbsp; &nbsp; 

Fit silicon to time-of-flight neutron powder data from SEPD at Argonne, with the Jorgensen-Von
Dreele profile.

The project is the catalogue case `pd-neut-tof_si-sepd`, the one the command line and the app's
Examples open.

In [2]:
import easydiffraction as edi

## Load the project

In [3]:
project_path = edi.download_data('pd-neut-tof_si-sepd', destination='data')
project = edi.Project.load(project_path)
structure = project.structures['si']
experiment = project.experiments['sepd']

Data 'pd-neut-tof_si-sepd' is at data/pd-neut-tof_si-sepd-d8aba2784c50/pd-neut-tof_si-sepd


The project marks what its fit frees. The fit below frees it in stages, so start with all of it
fixed.

In [4]:
for parameter in project.free_parameters:
    parameter.free = False

## Estimate the background

Start the background from an automatic estimate. The default method, arPLS, fits a smooth curve
under the pattern that ignores the peaks, and places 14 points along it.

In [5]:
experiment.background.auto_estimate(n_points=14)

In [6]:
project.display.pattern('sepd')

View(html=<693,289 characters>, height=480)

## Perform the analysis

The fit follows diffraction-lib's four cycles for this case, then frees the background once more
so it ends with the case's free set.

### Cell, scale and time-of-flight calibration

In [7]:
structure.cell.length_a.free = True
experiment.linked_structures['si'].scale.free = True
experiment.instrument.calib_d_to_tof_offset.free = True

In [8]:
project.display.parameters.free()

#,datablock,category,entry,parameter,value,s.u.,min,max,units
1,si,cell,,length_a,5.431,,0,30,Å
2,sepd,instrument,,calib_d_to_tof_offset,0,,-inf,inf,µs
3,sepd,linked_structure,si,scale,600,,-inf,inf,


In [9]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-tof_si-sepd, 5600 pts fitted / 5600 loaded, 3 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       52.36       204.71
      1      0.008     19.80        29.28   85.7% ↓
      2      0.014     10.94         8.93   69.5% ↓
      3      0.020     10.45         8.16   8.7% ↓
      4      0.025     10.45         8.16   
      5      0.038     10.45         8.16   
  done · converged · 5 iters · reduced χ² 8.16 · Rwp 10.45 % · 0.05 s


View(html=<79,940 characters>, height=376)

In [10]:
project.display.pattern('sepd')

View(html=<693,365 characters>, height=480)

### Background

The estimated points are a starting point; fit them now that the peaks are in place.

In [11]:
for point in experiment.background:
    point.intensity.free = True

In [12]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-tof_si-sepd, 5600 pts fitted / 5600 loaded, 17 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       10.45         8.18
      1      0.014      7.45         4.16   49.1% ↓
      2      0.028      7.45         4.16   
  done · converged · 2 iters · reduced χ² 4.16 · Rwp 7.45 % · 0.04 s


View(html=<94,250 characters>, height=376)

In [13]:
project.display.pattern('sepd')

View(html=<693,003 characters>, height=480)

### Gaussian widths

The background stays free while the widths join. diffraction-lib fixes it for this step and
leaves it fixed; here the fit reaches the same result with it free, as every other case does.
The Gaussian terms come first.

In [14]:
experiment.peak.broad_gauss_sigma_0.free = True
experiment.peak.broad_gauss_sigma_1.free = True

In [15]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-tof_si-sepd, 5600 pts fitted / 5600 loaded, 19 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        7.45         4.16
      1      0.017      7.42         4.13   0.8% ↓
      2      0.025      7.42         4.13   
      3      0.032      7.42         4.13   
      4      0.048      7.42         4.13   
  done · converged · 4 iters · reduced χ² 4.13 · Rwp 7.42 % · 0.06 s


View(html=<96,288 characters>, height=376)

In [16]:
project.display.pattern('sepd')

View(html=<692,723 characters>, height=480)

### Lorentzian width

Then the Lorentzian term.

In [17]:
experiment.peak.broad_lorentz_gamma_1.free = True

In [18]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-tof_si-sepd, 5600 pts fitted / 5600 loaded, 20 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        7.42         4.13
      1      0.015      7.28         3.98   3.7% ↓
      2      0.022      7.28         3.97   
      3      0.038      7.28         3.97   
  done · converged · 3 iters · reduced χ² 3.97 · Rwp 7.28 % · 0.05 s


View(html=<97,530 characters>, height=376)

In [19]:
project.display.pattern('sepd')

View(html=<693,022 characters>, height=480)

### Displacement

In [20]:
structure.atom_sites['Si'].adp_iso.free = True

In [21]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-tof_si-sepd, 5600 pts fitted / 5600 loaded, 21 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        7.28         3.98
      1      0.017      7.28         3.97   0.2% ↓
      2      0.025      7.28         3.97   
      3      0.040      7.28         3.97   
  done · converged · 3 iters · reduced χ² 3.97 · Rwp 7.28 % · 0.05 s


View(html=<98,423 characters>, height=376)

In [22]:
project.display.pattern('sepd')

View(html=<692,889 characters>, height=480)

### Peak decay

The back-to-back exponential decay terms β₀ and β₁.

In [23]:
experiment.peak.decay_beta_0.free = True
experiment.peak.decay_beta_1.free = True

In [24]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-tof_si-sepd, 5600 pts fitted / 5600 loaded, 23 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        7.28         3.97
      1      0.015      7.18         3.87   2.5% ↓
      2      0.023      7.18         3.86   0.2% ↓
      3      0.030      7.18         3.86   
      4      0.037      7.18         3.86   
      5      0.052      7.18         3.86   
  done · converged · 5 iters · reduced χ² 3.86 · Rwp 7.18 % · 0.06 s


View(html=<100,513 characters>, height=376)

In [25]:
project.display.pattern('sepd')

View(html=<692,845 characters>, height=480)

In [26]:
project.display.fit.correlations()

,1,2,3,4
1 si.cell.length_a,,,,
2 sepd.instrument.calib_d_to_tof_offset,-0.88,,,
3 sepd.peak.decay_beta_0,0.69,-0.49,,
4 sepd.peak.decay_beta_1,-0.48,0.68,-0.43,


In [27]:
project.display.structure('si')

View(html=<115,462 characters>, height=660)

## Save the project

In [28]:
project.save_as('pd-neut-tof_si-sepd')
project.report.save_html('pd-neut-tof_si-sepd.html')

Saving project to 'pd-neut-tof_si-sepd'
Saving report to 'pd-neut-tof_si-sepd.html'


## Your own measured data

Measured data can also come from a file on your own disk. `add_from_data_path` reads a file of
time-of-flight, intensity and uncertainty columns into a new experiment, here in a project of its
own; the experiment then needs its instrument, profile and linked structure, as above. The file
here is SEPD's pattern from the collection, saved to disk first.

In [29]:
data_path = edi.download_data('pd-neut-tof_si-sepd-sepd-data', destination='data')
own = edi.Project(name='own_sepd')
own.experiments.add_from_data_path(
    name='mine',
    data_path=data_path,
    sample_form='powder',
    beam_mode='time-of-flight',
    radiation_probe='neutron',
)
own.display.pattern('mine')

Data 'pd-neut-tof_si-sepd-sepd-data' is at data/pd-neut-tof_si-sepd-sepd.xye


View(html=<296,398 characters>, height=480)